In [1]:
import pysam
import pandas as pd

In [2]:
wk_dir = "/working/lab_olgak/lijunX/OV_meBRCA1_Nanopore"
#SampleID = "134169_6606_T2"
#SampleID = "SOLACE2_0126S1_25285_T2"
SampleID = "SFRC01428S1_22235_T2"

In [3]:
vcf_file = wk_dir + "/output/"+SampleID+"/deepvariants/"+SampleID+".vcf.gz"
deepvariants = pysam.VariantFile(vcf_file)
deepvariantsFile = deepvariants.fetch("chr17", 43128051-50,43128516+50)


bamfile_path = wk_dir+"/output/"+SampleID+"/"+SampleID+"_filtered_mapped.bam"
for variants in deepvariantsFile:
    ref_allele, alt_allele = variants.alleles
    print(alt_allele)

C
A
GAGGTCTGCAGCTTCACTCCTGAAGCCAGCGAGACCACGAACCCACCAGGAGGAATGAACAACTCCAGATGGGAGGAACGAACAAACTCCAGACATGCCGCCTTAAGAGCTGTAACACTCACCGCAA
T
C
G
G


In [4]:
%%capture cap --no-stderr
vcf_file = wk_dir + "/output/"+SampleID+"/deepvariants/"+SampleID+".vcf.gz"
deepvariants = pysam.VariantFile(vcf_file)
deepvariantsFile = deepvariants.fetch("chr17", 43128051-50,43128516+50)


bamfile_path = wk_dir+"/output/"+SampleID+"/"+SampleID+"_filtered_mapped.bam"

match = list()
discordant = list()
for variants in deepvariantsFile:
    ref_allele, alt_allele = variants.alleles
    if len(alt_allele) >= 50:
        print("chrom:", variants.chrom, " start:", variants.start, " stop:", variants.stop)
        print("alt:",alt_allele)
        bamfile =  pysam.AlignmentFile(bamfile_path, "rb").fetch("chr17", variants.start-200,variants.stop+200)
        for reads in bamfile:
            read_pos = reads.get_reference_positions(full_length =True)
            pos_1 = read_pos.index(variants.start) if variants.start in read_pos else 1
            pos_2 = read_pos.index(variants.start+2) if variants.start+2 in read_pos else 2
            if read_pos[pos_1-1:pos_2].count(None)>=50:
                discordant.append(reads.query_name)
            else:
                match.append(reads.query_name)

print("match:", ",".join(match))
print("discordant:",",".join(discordant))


In [5]:
with open(wk_dir+"/output/"+SampleID+"/"+SampleID+"_LTR_readID.txt", 'w') as f:
    f.write(cap.stdout)